In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import poisson

files = sorted(Path("../data/raw").glob("seriea_*.csv"))
df = pd.concat([pd.read_csv(f).assign(Season=f.stem[-4:]) for f in files], ignore_index=True)
df = df[["Season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]].dropna()
df[["FTHG", "FTAG"]] = df[["FTHG", "FTAG"]].astype(int)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed")
df = df.sort_values("Date").reset_index(drop=True)

C:\Users\Utente\AppData\Local\Temp\ipykernel_20848\1612280969.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = pd.concat([pd.read_csv(f).assign(Season=f.stem[-4:]) for f in files], ignore_index=True)


In [2]:
media = df["FTHG"].mean()
reale = df["FTHG"].value_counts(normalize=True).sort_index()
pd.DataFrame({"reale": reale, "poisson": poisson.pmf(reale.index, media)}).round(3)

,reale,poisson
FTHG,,
0,0.235,0.227
1,0.332,0.336
2,0.245,0.250
3,0.120,0.124
4,0.046,0.046
5,0.016,0.014
6,0.005,0.003
7,0.001,0.001
8,0.000,0.000


In [3]:
def stima_forze(train):
    casa = train.rename(columns={"HomeTeam": "team", "FTHG": "gf", "FTAG": "gs"})
    trasf = train.rename(columns={"AwayTeam": "team", "FTAG": "gf", "FTHG": "gs"})
    tutte = pd.concat([casa, trasf])[["team", "gf", "gs"]]
    medie = tutte.groupby("team").mean()
    media_gol = tutte["gf"].mean()
    forze = pd.DataFrame({"att": medie["gf"] / media_gol, "dif": medie["gs"] / media_gol})
    return forze, train["FTHG"].mean(), train["FTAG"].mean()

In [4]:
NEOPROMOSSA = (1.0, 1.0)

def prob_partita(casa, trasf, forze, media_casa, media_trasf, max_gol=10):
    att_c, dif_c = forze.loc[casa] if casa in forze.index else NEOPROMOSSA
    att_t, dif_t = forze.loc[trasf] if trasf in forze.index else NEOPROMOSSA
    lam = media_casa * att_c * dif_t
    mu = media_trasf * att_t * dif_c
    gol = np.arange(max_gol + 1)
    matrice = np.outer(poisson.pmf(gol, lam), poisson.pmf(gol, mu))
    p_casa = np.tril(matrice, -1).sum()
    p_pari = np.trace(matrice)
    p_trasf = np.triu(matrice, 1).sum()
    return p_casa, p_pari, p_trasf, matrice

In [5]:
train = df[df["Season"].isin(["2425", "2526", "2627"])]
forze, mc, mt = stima_forze(train)
print(forze.sort_values("att", ascending=False).round(2).head(8))

p1, px, p2, m = prob_partita("Inter", "Milan", forze, mc, mt)
print(f"\nInter-Milan → 1: {p1:.1%}   X: {px:.1%}   2: {p2:.1%}")
pd.DataFrame(m[:5, :5]).round(3)

            att   dif
team                 
Inter      1.75  0.74
Frosinone  1.70  1.06
Roma       1.31  0.68
Atalanta   1.31  0.83
Milan      1.22  0.81
Napoli     1.21  0.69
Juventus   1.20  0.72
Como       1.18  0.81

Inter-Milan → 1: 56.1%   X: 22.1%   2: 21.8%


,0,1,2,3,4
0,0.048,0.054,0.030,0.011,0.003
1,0.092,0.103,0.057,0.021,0.006
2,0.089,0.099,0.055,0.020,0.006
3,0.057,0.063,0.035,0.013,0.004
4,0.027,0.030,0.017,0.006,0.002


In [6]:
stagioni = sorted(df["Season"].unique())
righe = []
for i, s in enumerate(stagioni):
    if i < 3:
        continue
    train = df[df["Season"].isin(stagioni[i-3:i])]
    forze, mc, mt = stima_forze(train)
    for row in df[df["Season"] == s].itertuples():
        p1, px, p2, _ = prob_partita(row.HomeTeam, row.AwayTeam, forze, mc, mt)
        righe.append({"idx": row.Index, "pH": p1, "pD": px, "pA": p2})

ris = df.join(pd.DataFrame(righe).set_index("idx"), how="inner")

In [7]:
test = ris[ris["Date"] >= "2023-07-01"]
probs = test[["pH", "pD", "pA"]].to_numpy()
probs = probs / probs.sum(axis=1, keepdims=True)

pred = np.array(["H", "D", "A"])[probs.argmax(axis=1)]
acc = (pred == test["FTR"]).mean()

col = test["FTR"].map({"H": 0, "D": 1, "A": 2}).to_numpy()
logloss = -np.log(probs[np.arange(len(test)), col]).mean()

freq = test["FTR"].value_counts(normalize=True)
logloss_base = -np.log(test["FTR"].map(freq)).mean()

print(f"Accuratezza Poisson: {acc:.1%}   (Elo: 53,0%)")
print(f"Log-loss Poisson: {logloss:.3f}   Baseline: {logloss_base:.3f}")
print("Pareggi previsti:", (pred == "D").sum(), "su", len(test))

Accuratezza Poisson: 51.5%   (Elo: 53,0%)
Log-loss Poisson: 0.993   Baseline: 1.084
Pareggi previsti: 0 su 1250


In [8]:
def stima_forze(train, prior=0):
    casa = train.rename(columns={"HomeTeam": "team", "FTHG": "gf", "FTAG": "gs"})
    trasf = train.rename(columns={"AwayTeam": "team", "FTAG": "gf", "FTHG": "gs"})
    tutte = pd.concat([casa, trasf])[["team", "gf", "gs"]]
    media_gol = tutte["gf"].mean()
    g = tutte.groupby("team").agg(n=("gf", "size"), gf=("gf", "sum"), gs=("gs", "sum"))
    forze = pd.DataFrame({
        "att": (g["gf"] + prior * media_gol) / (g["n"] + prior) / media_gol,
        "dif": (g["gs"] + prior * media_gol) / (g["n"] + prior) / media_gol,
    })
    return forze, train["FTHG"].mean(), train["FTAG"].mean()

In [9]:
train = df[df["Season"].isin(["2425", "2526", "2627"])]
prima, _, _ = stima_forze(train, prior=0)
dopo, _, _ = stima_forze(train, prior=15)
squadre = ["Inter", "Frosinone", "Roma", "Napoli"]
pd.concat([prima.loc[squadre], dopo.loc[squadre]], axis=1, keys=["senza", "con shrinkage"]).round(2)

senza       con shrinkage      
            att   dif           att   dif
team                                     
Inter      1.75  0.74          1.64  0.78
Frosinone  1.70  1.06          1.29  1.03
Roma       1.31  0.68          1.26  0.73
Napoli     1.21  0.69          1.18  0.73

In [10]:
def previsioni(df, prior, aggiorna=True, finestra=3):
    stagioni = sorted(df["Season"].unique())
    righe = []
    for i, s in enumerate(stagioni):
        if i < finestra:
            continue
        passate = df[df["Season"].isin(stagioni[i-finestra:i])]
        corrente = df[df["Season"] == s]
        for giorno, partite in corrente.groupby("Date"):
            if aggiorna:
                train = pd.concat([passate, corrente[corrente["Date"] < giorno]])
            else:
                train = passate
            forze, mc, mt = stima_forze(train, prior)
            for row in partite.itertuples():
                p1, px, p2, _ = prob_partita(row.HomeTeam, row.AwayTeam, forze, mc, mt)
                righe.append({"idx": row.Index, "pH": p1, "pD": px, "pA": p2})
    return df.join(pd.DataFrame(righe).set_index("idx"), how="inner")

def valuta(ris, inizio, fine="2100-01-01"):
    t = ris[(ris["Date"] >= inizio) & (ris["Date"] < fine)]
    probs = t[["pH", "pD", "pA"]].to_numpy()
    probs = probs / probs.sum(axis=1, keepdims=True)
    pred = np.array(["H", "D", "A"])[probs.argmax(axis=1)]
    col = t["FTR"].map({"H": 0, "D": 1, "A": 2}).to_numpy()
    acc = (pred == t["FTR"]).mean()
    logloss = -np.log(probs[np.arange(len(t)), col]).mean()
    return acc, logloss

In [11]:
risultati = []
for prior in [2, 5, 10, 20, 30, 50]:
    ris = previsioni(df, prior)
    acc, ll = valuta(ris, "2019-07-01", "2023-07-01")
    risultati.append({"prior": prior, "acc_tuning": acc, "logloss_tuning": ll})

tab = pd.DataFrame(risultati).sort_values("logloss_tuning")
tab

,prior,acc_tuning,logloss_tuning
1,5,0.535526,0.986842
2,10,0.536184,0.986849
0,2,0.534868,0.987989
3,20,0.535526,0.988475
4,30,0.534868,0.990783
5,50,0.535526,0.995923


In [12]:
best = int(tab.iloc[0]["prior"])
varianti = [
    ("Poisson base", previsioni(df, 0, aggiorna=False)),
    ("+ shrinkage", previsioni(df, best, aggiorna=False)),
    ("+ aggiornamento", previsioni(df, best, aggiorna=True)),
]
print(f"prior scelto: {best}")
for nome, ris in varianti:
    acc, ll = valuta(ris, "2023-07-01")
    print(f"{nome:16s} accuratezza {acc:.1%}   log-loss {ll:.3f}")

prior scelto: 5
Poisson base     accuratezza 51.5%   log-loss 0.993
+ shrinkage      accuratezza 51.1%   log-loss 0.993
+ aggiornamento  accuratezza 52.6%   log-loss 0.982


In [13]:
PRIOR = 5
corrente = df[df["Season"] == "2627"]
squadre = sorted(set(corrente["HomeTeam"]) | set(corrente["AwayTeam"]))
idx = {s: i for i, s in enumerate(squadre)}

train = df[df["Season"].isin(["2324", "2425", "2526", "2627"])]
forze, mc, mt = stima_forze(train, PRIOR)

giocate = set(zip(corrente["HomeTeam"], corrente["AwayTeam"]))
restanti = [(c, t) for c in squadre for t in squadre if c != t and (c, t) not in giocate]
print(len(squadre), "squadre,", len(corrente), "giocate,", len(restanti), "da giocare")

20 squadre, 110 giocate, 270 da giocare


In [14]:
punti0 = np.zeros(len(squadre))
dr0 = np.zeros(len(squadre))
for r in corrente.itertuples():
    c, t = idx[r.HomeTeam], idx[r.AwayTeam]
    punti0[c] += 3 if r.FTR == "H" else 1 if r.FTR == "D" else 0
    punti0[t] += 3 if r.FTR == "A" else 1 if r.FTR == "D" else 0
    dr0[c] += r.FTHG - r.FTAG
    dr0[t] += r.FTAG - r.FTHG

pd.DataFrame({"punti": punti0, "diff_reti": dr0}, index=squadre).astype(int).sort_values(
    ["punti", "diff_reti"], ascending=False)

,punti,diff_reti
Lazio,29,16
Inter,28,15
Roma,27,21
Milan,24,10
Como,23,8
Frosinone,20,9
Juventus,18,3
Bologna,17,2
Cagliari,16,-5
Napoli,15,4


In [15]:
lam = np.array([mc * forze.at[c, "att"] * forze.at[t, "dif"] for c, t in restanti])
mu = np.array([mt * forze.at[t, "att"] * forze.at[c, "dif"] for c, t in restanti])

In [16]:
N = 10000
rng = np.random.default_rng(42)
gc = rng.poisson(lam, size=(N, len(restanti)))
gt = rng.poisson(mu, size=(N, len(restanti)))

punti = np.tile(punti0, (N, 1))
dr = np.tile(dr0, (N, 1))
for j, (c, t) in enumerate(restanti):
    ic, it = idx[c], idx[t]
    pari = gc[:, j] == gt[:, j]
    punti[:, ic] += 3 * (gc[:, j] > gt[:, j]) + pari
    punti[:, it] += 3 * (gt[:, j] > gc[:, j]) + pari
    dr[:, ic] += gc[:, j] - gt[:, j]
    dr[:, it] += gt[:, j] - gc[:, j]

chiave = punti + dr / 1000 + rng.random(punti.shape) / 1e6
posizione = (-chiave).argsort(axis=1).argsort(axis=1) + 1

In [17]:
tabella = pd.DataFrame({
    "punti_ora": punti0.astype(int),
    "punti_finali": punti.mean(axis=0),
    "scudetto": (posizione == 1).mean(axis=0),
    "top4": (posizione <= 4).mean(axis=0),
    "retrocessione": (posizione >= 18).mean(axis=0),
}, index=squadre).sort_values("punti_finali", ascending=False)

tabella.style.format({"punti_finali": "{:.1f}", "scudetto": "{:.1%}",
                      "top4": "{:.1%}", "retrocessione": "{:.1%}"})

,punti_ora,punti_finali,scudetto,top4,retrocessione
Inter,28,88.1,87.4%,99.8%,0.0%
Roma,27,76.4,8.1%,87.5%,0.0%
Milan,24,71.1,2.2%,61.4%,0.0%
Lazio,29,70.0,1.3%,53.5%,0.0%
Como,23,67.9,0.7%,40.6%,0.0%
Juventus,18,65.4,0.2%,25.7%,0.0%
Napoli,15,62.1,0.1%,14.2%,0.0%
Atalanta,14,61.4,0.1%,11.1%,0.0%
Bologna,17,58.8,0.0%,5.7%,0.0%
Frosinone,20,50.7,0.0%,0.4%,0.2%
